In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, ReLU
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt

In [ ]:
# GPU memory setup
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(len(gpus), "Physical GPUs,", len(tf.config.experimental.list_logical_devices('GPU')), "Logical GPUs")
    except RuntimeError as e:
        print(e)

In [ ]:
# === Global Configuration ===
EXECUTE_TRAINING = True
TRANSFER_LEARNING = False
SHUFFLE_DATA = False
MODEL_DIR = 'Model_architecture/FCNN_linear/'
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# === Load and Preprocess Datasets ===
print("Loading datasets...")
X_data = np.load('Dataset/CFD/dataset_CL0_CM0_M_AoA.npy')[:, 0, :]
Y_data = np.load('Dataset/Volterra/kernels_CL_CM_linear.npy')
Y_data = np.transpose(Y_data, (0, 2, 1))
Y_data, Y_temp = Y_data[:, :, :-1], np.expand_dims(Y_data[:, :, -1], axis=2)
X_data = np.expand_dims(X_data, axis=1)

n_samples, n_input_pts, n_input_vars = X_data.shape
_, n_output_channels, n_timesteps = Y_data.shape

# Normalize input/output
scaler_X = MinMaxScaler(feature_range=(-1, 1))
X_data_flat = X_data.reshape(-1, n_input_vars)
X_scaled = scaler_X.fit_transform(X_data_flat).reshape(n_samples, n_input_pts, n_input_vars)

scaler_Y = MinMaxScaler(feature_range=(-1, 1))
Y_data_flat = Y_data.reshape(-1, n_timesteps)
Y_scaled = scaler_Y.fit_transform(Y_data_flat).reshape(n_samples, n_output_channels, n_timesteps)

# Train/Validation/Test Split
print("Splitting dataset...")
train_ratio, test_ratio, val_ratio = 0.58, 0.20, 0.20
X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y_scaled, test_size=(1 - train_ratio), shuffle=SHUFFLE_DATA)
X_val, X_test, Y_val, Y_test = train_test_split(X_test, Y_test, test_size=(test_ratio / (test_ratio + val_ratio)), shuffle=SHUFFLE_DATA)


In [ ]:
# === Model Definition ===
def build_model(hp):
    inp_shape = (X_scaled.shape[1], X_scaled.shape[2])
    inputs = Input(shape=inp_shape)
    x = inputs
    prelu = ReLU(max_value=None, negative_slope=0.02)

    for i in range(hp.Int('layers', 3, 7)):
        units = hp.Int(f'units_{i}', 32, 256, step=16)
        x = Dense(units=units, kernel_initializer='glorot_normal', activation=prelu)(x)

    output_CL = Dense(n_timesteps, kernel_regularizer='l2', activation='tanh', name='kernels_CL_linear_pred')(x)
    output_CM = Dense(n_timesteps, kernel_regularizer='l2', activation='tanh', name='kernels_CM_linear_pred')(x)
    model = tf.keras.Model(inputs=inputs, outputs=[output_CL, output_CM])

    model.compile(
        loss={'kernels_CL_linear_pred': 'mse', 'kernels_CM_linear_pred': 'mse'},
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics={'kernels_CL_linear_pred': 'mse', 'kernels_CM_linear_pred': 'mse'}
    )
    return model

# === Training Phase ===
if EXECUTE_TRAINING:
    print("Setting up tuner and early stopping...")
    tuner = kt.BayesianOptimization(
        build_model,
        objective=kt.Objective('val_kernels_CL_linear_pred_loss', direction='min'),
        max_trials=50,
        overwrite=True,
        directory=MODEL_DIR + 'BayesianOptimization_results',
        project_name="Transonic_wing"
    )

    early_stop = EarlyStopping(monitor='val_kernels_CL_linear_pred_loss', patience=50, restore_best_weights=True)
    tensorboard_cb = tf.keras.callbacks.TensorBoard(MODEL_DIR + 'logs_optmization')

    tuner.search(
        x=X_train,
        y={
            'kernels_CL_linear_pred': Y_train[:, :1, :],
            'kernels_CM_linear_pred': Y_train[:, 1:, :]
        },
        validation_data=(
            X_test,
            {
                'kernels_CL_linear_pred': Y_test[:, :1, :],
                'kernels_CM_linear_pred': Y_test[:, 1:, :]
            }
        ),
        batch_size=5,
        callbacks=[early_stop, tensorboard_cb],
        epochs=500,
        verbose=0
    )

    best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
    model = tuner.hypermodel.build(best_hp)
    early_stop_final = EarlyStopping(monitor='val_loss', patience=300, restore_best_weights=True)

    print("Training best model...")
    history = model.fit(
        x=X_train,
        y={
            'kernels_CL_linear_pred': Y_train[:, :1, :],
            'kernels_CM_linear_pred': Y_train[:, 1:, :]
        },
        validation_data=(
            X_test,
            {
                'kernels_CL_linear_pred': Y_test[:, :1, :],
                'kernels_CM_linear_pred': Y_test[:, 1:, :]
            }
        ),
        batch_size=5,
        callbacks=[early_stop_final],
        epochs=10000,
        verbose=2
    )

    print("Saving trained model...")
    model.save(MODEL_DIR + 'neural_network_model.h5')
    model.save_weights(MODEL_DIR + 'neural_network_weights.h5')

# === Transfer Learning ===
if TRANSFER_LEARNING:
    print("Loading pretrained model...")
    model = load_model(MODEL_DIR + 'neural_network_model.h5')
    model.load_weights(MODEL_DIR + 'neural_network_weights.h5')
    model.summary()

In [ ]:
# === Prediction on Validation Set ===
print("Predicting on validation set...")
Y_pred_CL, Y_pred_CM = model.predict(X_val, batch_size=1)

# Rescale predictions and ground truth
X_val_flat = X_val.reshape(-1, X_val.shape[2])
X_val_inv = scaler_X.inverse_transform(X_val_flat).reshape(X_val.shape)

Y_val_flat = Y_val.reshape(-1, Y_val.shape[2])
Y_val_inv = scaler_Y.inverse_transform(Y_val_flat).reshape(Y_val.shape)

Y_pred_CL_flat = Y_pred_CL.reshape(-1, Y_pred_CL.shape[2])
Y_pred_CL_inv = scaler_Y.inverse_transform(Y_pred_CL_flat).reshape(Y_pred_CL.shape)

Y_pred_CM_flat = Y_pred_CM.reshape(-1, Y_pred_CM.shape[2])
Y_pred_CM_inv = scaler_Y.inverse_transform(Y_pred_CM_flat).reshape(Y_pred_CM.shape)

# Add last timestep = 0
zeros = np.zeros((Y_pred_CL_inv.shape[0], Y_pred_CL_inv.shape[1], 1))
Y_pred_CL_final = np.concatenate((Y_pred_CL_inv, zeros), axis=2)
Y_pred_CM_final = np.concatenate((Y_pred_CM_inv, zeros), axis=2)

np.save('Dataset/Predictions/kernels_CL_linear_pred.npy', Y_pred_CL_final[:, 0, :])
np.save('Dataset/Predictions/kernels_CM_linear_pred.npy', Y_pred_CM_final[:, 0, :])


In [ ]:
# === Grid Prediction (e.g., 0.74 < M < 0.84 and 0 < AoA < 5) ===
print("Generating predictions on AoA-Mach sweep grid...")
Mach_vals = np.arange(0.74, 0.86, 0.002)
AoA_vals = np.arange(0, 6, 0.1)
Mach_grid, AoA_grid = np.meshgrid(Mach_vals, AoA_vals)
Mach_AoA = np.column_stack((Mach_grid.ravel(), AoA_grid.ravel()))
CL0_CM0 = np.load('Dataset/Volterra/kernels_CL0_CM0_pred_074M084_0AoA5.npy')

X_exp = np.column_stack((Mach_AoA[:, 0], Mach_AoA[:, 1], CL0_CM0[:, 0], CL0_CM0[:, 1]))
X_exp = scaler_X.transform(X_exp).reshape(-1, 1, 4)
Y_grid_CL, Y_grid_CM = model.predict(X_exp, batch_size=1)

Y_grid_CL = scaler_Y.inverse_transform(Y_grid_CL.reshape(-1, Y_grid_CL.shape[2])).reshape(Y_grid_CL.shape)
Y_grid_CM = scaler_Y.inverse_transform(Y_grid_CM.reshape(-1, Y_grid_CM.shape[2])).reshape(Y_grid_CM.shape)

zeros = np.zeros((Y_grid_CL.shape[0], Y_grid_CL.shape[1], 1))
Y_grid_CL = np.concatenate((Y_grid_CL, zeros), axis=2)
Y_grid_CM = np.concatenate((Y_grid_CM, zeros), axis=2)

np.save('Dataset/Predictions/kernels_CL_linear_pred_074M084_0AoA5.npy', Y_grid_CL[:, 0, :])
np.save('Dataset/Predictions/kernels_CM_linear_pred_074M084_0AoA5.npy', Y_grid_CM[:, 0, :])
